# Module 02 — Lesson 06
# Dictionaries

---

> If lists are the workhorse of Python, dictionaries are the data model.

A list lets you store values and access them by **position** — `scores[0]`, `scores[1]`. That works fine for a sequence of identical things, but falls apart when you want to store a *record*: a student with a name, age, score, and city. Which index is the name? Which is the age? You'd have to remember, and it's easy to get wrong.

A dictionary stores values and retrieves them by **name** — `student["name"]`, `student["score"]`. The name is always right there.

This is how virtually all structured data in the real world is represented:
- Every row in a Pandas DataFrame is a dictionary under the hood
- Every JSON API response is a dictionary (or a list of dictionaries)
- Every CSV header + row pair is a dictionary
- Every configuration file (`config.json`, environment variables) is a dictionary

Master dictionaries here and you'll immediately recognise their structure everywhere.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Create dictionaries using literal syntax, `dict()`, `zip()`, and comprehensions
- Access values safely using `d[key]` and `d.get(key, default)`
- Add, update, and delete entries; merge two dicts
- Iterate over `.keys()`, `.values()`, and `.items()`
- Work with nested dictionaries representing real data records
- Use dict comprehensions to transform and filter mappings
- Apply classic patterns: frequency counter, groupBy, label encoder, dict inversion

---
## 1. What Is a Dictionary?

A dictionary is a collection of **key-value pairs**.

```
  {  "name"  :  "Priya"  ,  "age"  :  20  ,  "score"  :  88.5  }
      key         value       key      value    key          value
```

**Rules for keys:**
- Must be **immutable** — strings, numbers, and tuples are valid; lists are not
- Must be **unique** — duplicate keys overwrite each other
- Can be any type (strings are most common)

**Values** can be anything: numbers, strings, lists, other dicts, functions.

In [ ]:
# Why dict is better than list for records

# ❌ List: you must remember which index means what
student_list = ["Priya", 20, 88.5, "Mumbai", "CS"]
print(f"Name: {student_list[0]}, Score: {student_list[2]}")  # error-prone

# ✅ Dict: self-describing, can access in any order
student_dict = {
    "name"  : "Priya",
    "age"   : 20,
    "score" : 88.5,
    "city"  : "Mumbai",
    "dept"  : "CS",
}
print(f"Name: {student_dict['name']}, Score: {student_dict['score']}")

# Basic properties
print(f"\nType    : {type(student_dict)}")
print(f"Length  : {len(student_dict)} key-value pairs")

---
## 2. Creating Dictionaries

In [ ]:
# Method 1: literal syntax {key: value, ...}
product = {
    "id"       : "P001",
    "name"     : "Wireless Earbuds",
    "price"    : 1299.0,
    "in_stock" : True,
    "tags"     : ["electronics", "audio"],   # values can be lists
}
print("Literal:", product)

# Method 2: dict() constructor with keyword arguments
city_info = dict(name="Mumbai", population=20_667_000, state="Maharashtra")
print("dict()  :", city_info)

# Method 3: empty dict, then fill it
record = {}
record["name"]  = "Priya"
record["score"] = 88
print("Built up:", record)

In [ ]:
# Method 4: from two parallel lists using zip()
# This is extremely common when loading CSV data column by column

keys   = ["name", "age", "score", "city"]
values = ["Priya",  20,    88.5,   "Mumbai"]

student = dict(zip(keys, values))
print("From zip:", student)

# Method 5: dict comprehension (covered in depth in Section 7)
squares = {n: n**2 for n in range(1, 6)}
print("Comprehension:", squares)

---
## 3. Accessing Values

Two ways — and you should know when to use each.

In [ ]:
student = {"name": "Priya", "age": 20, "score": 88.5, "city": "Mumbai"}

# Method 1: d[key] — raises KeyError if key doesn't exist
print(student["name"])    # Priya
print(student["score"])   # 88.5

# KeyError when key is missing
try:
    print(student["email"])   # KeyError!
except KeyError as e:
    print(f"KeyError: {e} — key does not exist")

In [ ]:
# Method 2: d.get(key) — returns None (or a default) if key is missing
# This is the SAFE way — use it when the key might not exist

print(student.get("name"))          # Priya
print(student.get("email"))         # None  — no crash
print(student.get("email", "N/A"))  # N/A   — custom default
print(student.get("age", 0))        # 20    — key exists, default ignored

# Applied: safely extract optional fields from user-submitted data
form_data = {"name": "Rohan", "score": 72}

name     = form_data.get("name", "Anonymous")
score    = form_data.get("score", 0)
city     = form_data.get("city", "Not specified")   # key missing — uses default
email    = form_data.get("email")                   # key missing — returns None

print(f"\n{name}: score={score}, city={city}, email={email!r}")

In [ ]:
# Membership check: 'in' tests keys, not values
print("name"  in student)    # True  — key exists
print("email" in student)    # False — key missing
print("Priya" in student)    # False — 'in' checks keys, not values!
print("Priya" in student.values())  # True — explicitly check values

# Guard pattern: only access if key exists
if "score" in student:
    grade = "Pass" if student["score"] >= 40 else "Fail"
    print(f"Grade: {grade}")

---
## 4. Modifying Dictionaries

In [ ]:
student = {"name": "Priya", "age": 20, "score": 88.5}

# Add a new key
student["city"] = "Mumbai"
print("After add   :", student)

# Update an existing key (same syntax)
student["score"] = 91.0
print("After update:", student)

# Delete a key — del raises KeyError if missing
del student["age"]
print("After del   :", student)

# pop(key) — removes AND returns the value; safe version with default
score = student.pop("score")
print(f"Popped score: {score}")
print("After pop   :", student)

# pop with default — won't crash if key is missing
email = student.pop("email", None)
print(f"Pop missing key with default: {email!r}")

In [ ]:
# update() — merge another dict into this one (existing keys are overwritten)
record = {"name": "Priya", "age": 20, "score": 88.5}
extras = {"city": "Mumbai", "dept": "CS", "score": 91.0}  # 'score' will overwrite

record.update(extras)
print("After update():", record)

In [ ]:
# Merging dicts — three ways
base    = {"name": "Priya", "age": 20}
overlay = {"score": 88.5, "city": "Mumbai"}

# Option A: {**d1, **d2} — unpack both (Python 3.5+)
merged_a = {**base, **overlay}
print("Unpack merge:", merged_a)

# Option B: d1 | d2 — pipe operator (Python 3.9+)
merged_b = base | overlay
print("Pipe merge  :", merged_b)

# Option C: .update() — modifies in place (no new dict created)
base.update(overlay)
print(".update()   :", base)

# Real use: add a derived column to a record
student = {"name": "Priya", "score": 88.5}
enriched = {**student, "grade": "A" if student["score"] >= 80 else "B"}
print("\nEnriched:", enriched)

---
## 5. Iterating Over Dictionaries

In [ ]:
student = {"name": "Priya", "age": 20, "score": 88.5, "city": "Mumbai"}

# Iterate over keys (default behaviour)
print("Keys (default):")
for key in student:
    print(f"  {key}")

# .keys() — explicit, same result
print("\n.keys():")
for key in student.keys():
    print(f"  {key}")

In [ ]:
# .values() — iterate over values only
print(".values():")
for value in student.values():
    print(f"  {value}")

print()
# .items() — iterate over (key, value) pairs — most useful
print(".items():")
for key, value in student.items():
    print(f"  {key:<8}: {value}")

In [ ]:
# Applied: print a formatted record card
def print_record(record, title="Record"):
    """Print a dict as a formatted card."""
    width = max(len(k) for k in record) + 2
    print(f"┌─ {title} " + "─" * (30 - len(title)) + "┐")
    for key, value in record.items():
        label = key.replace("_", " ").title()
        print(f"│  {label:<{width}}: {value}")
    print("└" + "─" * 34 + "┘")

student = {"name": "Priya Sharma", "age": 20, "score": 88.5,
           "city": "Mumbai", "dept": "CS", "is_enrolled": True}
print_record(student, "Student Profile")

In [ ]:
# Applied: aggregate stats across a list of records
students = [
    {"name": "Priya",  "score": 88, "dept": "CS"},
    {"name": "Rohan",  "score": 72, "dept": "ME"},
    {"name": "Meera",  "score": 95, "dept": "CS"},
    {"name": "Karan",  "score": 61, "dept": "IT"},
    {"name": "Ananya", "score": 38, "dept": "ME"},
    {"name": "Dev",    "score": 77, "dept": "CS"},
]

# Compute totals per department
dept_scores = {}   # dept → list of scores
for s in students:
    dept = s["dept"]
    if dept not in dept_scores:
        dept_scores[dept] = []
    dept_scores[dept].append(s["score"])

print(f"{'Dept':<6}  {'Scores':<25}  {'Avg':>6}")
print("─" * 44)
for dept, scores in sorted(dept_scores.items()):
    avg = sum(scores) / len(scores)
    print(f"{dept:<6}  {str(scores):<25}  {avg:>6.1f}")

---
## 6. Nested Dictionaries

A dict whose values are themselves dicts. This is how JSON data is structured, and how complex records are represented.

In [ ]:
# A student record with nested sub-documents
student = {
    "id"       : 1001,
    "name"     : "Priya Sharma",
    "contact"  : {
        "email" : "priya@example.com",
        "phone" : "9876543210",
        "city"  : "Mumbai",
    },
    "academics": {
        "dept"  : "CS",
        "year"  : 2,
        "scores": {"maths": 88, "science": 92, "english": 79},
    },
}

# Access nested values with chained []
print(student["name"])
print(student["contact"]["city"])
print(student["academics"]["scores"]["maths"])

# Compute average score from nested dict
scores = student["academics"]["scores"]
avg    = sum(scores.values()) / len(scores)
print(f"Average score: {avg:.1f}")

In [ ]:
# Safe access for deeply nested keys — chained .get()
# If any level is missing, .get() returns None instead of crashing

incomplete = {
    "id": 1002,
    "name": "Rohan Verma",
    # 'contact' key is missing entirely
}

# ❌ Crashes if any intermediate key is missing
try:
    email = incomplete["contact"]["email"]
except KeyError as e:
    print(f"KeyError: {e}")

# ✅ Safe chained .get()
email = incomplete.get("contact", {}).get("email", "Not provided")
print(f"Email (safe): {email}")

# Applied to the full student dict
maths_score = student.get("academics", {}).get("scores", {}).get("maths", None)
print(f"Maths score (safe): {maths_score}")

In [ ]:
# Database-like lookup: dict of dicts keyed by ID
# This is how you'd simulate a database table in memory

database = {
    1001: {"name": "Priya",  "score": 88, "dept": "CS"},
    1002: {"name": "Rohan",  "score": 72, "dept": "ME"},
    1003: {"name": "Meera",  "score": 95, "dept": "CS"},
    1004: {"name": "Karan",  "score": 61, "dept": "IT"},
}

def lookup_student(student_id):
    """Return student record by ID, or None if not found."""
    return database.get(student_id)

for sid in [1002, 1099]:   # 1099 doesn't exist
    record = lookup_student(sid)
    if record:
        print(f"ID {sid}: {record['name']} — score {record['score']}")
    else:
        print(f"ID {sid}: not found")

---
## 7. Dictionary Comprehensions

```python
{key_expr: value_expr  for item in iterable  if condition}
```

Same idea as list comprehensions, but produces a dict.

In [ ]:
# Transform: square each number
squares = {n: n**2 for n in range(1, 8)}
print("Squares:", squares)

# Transform: build a lookup from a list of records
students = [
    {"id": 1001, "name": "Priya",  "score": 88},
    {"id": 1002, "name": "Rohan",  "score": 72},
    {"id": 1003, "name": "Meera",  "score": 95},
]

# Build id → name mapping for fast lookup
id_to_name = {s["id"]: s["name"] for s in students}
print("ID lookup:", id_to_name)
print(f"Who is 1002? {id_to_name.get(1002, 'Unknown')}")

In [ ]:
# Filter: keep only key-value pairs matching a condition
scores = {"Priya": 88, "Rohan": 42, "Meera": 95, "Karan": 61, "Ananya": 38}

passing  = {name: score for name, score in scores.items() if score >= 40}
failing  = {name: score for name, score in scores.items() if score < 40}
high_ach = {name: score for name, score in scores.items() if score >= 80}

print("Passing :", passing)
print("Failing :", failing)
print("High (80+):", high_ach)

In [ ]:
# Transform keys and values simultaneously
raw_scores = {"  PRIYA  ": "88", "ROHAN": "72", "meera ": "95"}

# Clean keys (strip + title) and convert values (str → int)
clean = {k.strip().title(): int(v) for k, v in raw_scores.items()}
print("Cleaned:", clean)

# Invert a dict (swap keys and values)
grade_map = {"A": 4.0, "B": 3.0, "C": 2.0, "D": 1.0, "F": 0.0}
gpa_to_grade = {gpa: letter for letter, gpa in grade_map.items()}
print("Grade map  :", grade_map)
print("Inverted   :", gpa_to_grade)

---
## 8. Classic Dictionary Patterns

These patterns appear over and over in data science. Learn to recognise them.

### 8.1 — Frequency Counter

In [ ]:
def count_frequencies(items):
    """Count how often each unique value appears."""
    freq = {}
    for item in items:
        freq[item] = freq.get(item, 0) + 1   # .get with default 0 is the key trick
    return dict(sorted(freq.items(), key=lambda x: x[1], reverse=True))


# Grade distribution
grades = ["A", "B", "A", "C", "B", "A", "F", "B", "C", "A",
          "B", "C", "A", "B", "D", "C", "A", "B", "A", "C"]

freq = count_frequencies(grades)
total = sum(freq.values())
print(f"Grade distribution (n={total}):")
for grade, count in freq.items():
    bar = "█" * int(count / total * 40)
    print(f"  {grade}: {count:>3}  {count/total*100:>5.1f}%  {bar}")

### 8.2 — GroupBy (Group Records by a Key)

In [ ]:
def group_by(records, key):
    """Group a list of dicts by the value of a given key."""
    groups = {}
    for record in records:
        group_val = record.get(key)
        if group_val not in groups:
            groups[group_val] = []
        groups[group_val].append(record)
    return groups


employees = [
    {"name": "Priya",  "dept": "Engineering", "salary": 85000},
    {"name": "Rohan",  "dept": "Marketing",   "salary": 62000},
    {"name": "Meera",  "dept": "Engineering", "salary": 92000},
    {"name": "Karan",  "dept": "HR",          "salary": 55000},
    {"name": "Ananya", "dept": "Marketing",   "salary": 70000},
    {"name": "Dev",    "dept": "Engineering", "salary": 78000},
    {"name": "Riya",   "dept": "HR",          "salary": 58000},
]

by_dept = group_by(employees, "dept")

print(f"{'Department':<15}  {'Headcount':>10}  {'Avg Salary':>12}  {'Members'}")
print("─" * 70)
for dept, members in sorted(by_dept.items()):
    salaries = [m["salary"] for m in members]
    avg      = sum(salaries) / len(salaries)
    names    = ", ".join(m["name"] for m in members)
    print(f"{dept:<15}  {len(members):>10}  ₹{avg:>10,.0f}  {names}")

### 8.3 — Label Encoder (Category Mapping)

In [ ]:
# In ML, we must convert text categories to numbers.
# A dict is the simplest label encoder.

def build_label_encoder(categories):
    """Map each unique category to an integer."""
    unique = sorted(set(categories))
    return {cat: idx for idx, cat in enumerate(unique)}


def encode(values, encoder, unknown=-1):
    """Encode a list of category strings to integers."""
    return [encoder.get(v, unknown) for v in values]


def decode(values, encoder):
    """Reverse: integer back to category string."""
    reverse = {v: k for k, v in encoder.items()}
    return [reverse.get(v, "?") for v in values]


cities = ["Mumbai", "Delhi", "Mumbai", "Pune", "Delhi", "Chennai", "Mumbai"]
encoder = build_label_encoder(cities)

print(f"Encoder : {encoder}")
encoded = encode(cities, encoder)
print(f"Encoded : {encoded}")
decoded = decode(encoded, encoder)
print(f"Decoded : {decoded}")
print(f"Match   : {cities == decoded}")

# Unseen category gets -1 (unknown)
new_cities = ["Mumbai", "Bangalore", "Delhi"]
print(f"\nNew data encoded: {encode(new_cities, encoder)}")

### 8.4 — Aggregation Table

In [ ]:
# Build a summary table: {group: {metric: value}}
# This is what pandas .groupby().agg() does under the hood.

def aggregate(records, group_key, value_key):
    """
    For each unique group_key value, compute
    count, sum, min, max, mean of value_key.
    """
    groups = {}
    for r in records:
        grp = r.get(group_key)
        val = r.get(value_key)
        if val is None:
            continue
        if grp not in groups:
            groups[grp] = []
        groups[grp].append(val)

    result = {}
    for grp, vals in groups.items():
        result[grp] = {
            "count": len(vals),
            "sum"  : sum(vals),
            "min"  : min(vals),
            "max"  : max(vals),
            "mean" : round(sum(vals) / len(vals), 1),
        }
    return result


exam_results = [
    {"name": "Priya",  "dept": "CS", "score": 88},
    {"name": "Rohan",  "dept": "ME", "score": 72},
    {"name": "Meera",  "dept": "CS", "score": 95},
    {"name": "Karan",  "dept": "IT", "score": 61},
    {"name": "Ananya", "dept": "ME", "score": 38},
    {"name": "Dev",    "dept": "CS", "score": 77},
    {"name": "Riya",   "dept": "IT", "score": 82},
]

summary = aggregate(exam_results, "dept", "score")

print(f"{'Dept':<6}  {'Count':>6}  {'Mean':>6}  {'Min':>5}  {'Max':>5}  {'Sum':>6}")
print("─" * 42)
for dept, stats in sorted(summary.items()):
    print(f"{dept:<6}  {stats['count']:>6}  {stats['mean']:>6}  "
          f"{stats['min']:>5}  {stats['max']:>5}  {stats['sum']:>6}")

---
## 9. Useful Dictionary Methods

A quick reference for the methods you'll use most.

In [ ]:
d = {"a": 1, "b": 2, "c": 3}

print(f"len(d)         : {len(d)}")
print(f"d.keys()       : {list(d.keys())}")
print(f"d.values()     : {list(d.values())}")
print(f"d.items()      : {list(d.items())}")
print(f"d.get('a')     : {d.get('a')}")
print(f"d.get('z', 0)  : {d.get('z', 0)}")
print(f"'a' in d       : {'a' in d}")
print(f"'z' in d       : {'z' in d}")

# setdefault: get value if key exists, else set and return default
d.setdefault("d", 99)   # d didn't exist, so it's set to 99
d.setdefault("a", 99)   # a already exists, no change
print(f"\nAfter setdefault('d', 99) and setdefault('a', 99):")
print(f"  d = {d}")

# Copy
d_copy = d.copy()       # shallow copy — safe for flat dicts
d_copy["a"] = 999
print(f"\nOriginal d['a'] after modifying copy: {d['a']}  ← unchanged")

# Clear
temp = {"x": 1, "y": 2}
temp.clear()
print(f"After clear(): {temp}")

---
## ⚠️ Common Mistakes

In [ ]:
# ── Mistake 1: KeyError instead of .get() ─────────────────────────────────────
record = {"name": "Priya", "score": 88}

# ❌ Crashes if 'email' is ever missing
try:
    print(record["email"])   # KeyError
except KeyError:
    print("KeyError: use .get() for optional fields")

# ✅ Safe for optional fields
email = record.get("email", "Not provided")
print(f"Email: {email}")

In [ ]:
# ── Mistake 2: Modifying a dict while iterating over it ───────────────────────
scores = {"Priya": 88, "Rohan": 42, "Meera": 95, "Ananya": 38}

# ❌ RuntimeError — can't add/remove keys during iteration
try:
    for name, score in scores.items():
        if score < 40:
            del scores[name]   # modifying dict while iterating!
except RuntimeError as e:
    print(f"RuntimeError: {e}")

# ✅ Collect keys to remove first, then delete
scores = {"Priya": 88, "Rohan": 42, "Meera": 95, "Ananya": 38}
to_remove = [name for name, score in scores.items() if score < 40]
for name in to_remove:
    del scores[name]
print(f"After removing < 40: {scores}")

In [ ]:
# ── Mistake 3: Duplicate keys silently overwrite ──────────────────────────────
# Python does NOT warn you — the last value wins

d = {"name": "Priya", "score": 88, "name": "Rohan"}  # duplicate 'name'!
print(f"Duplicate key result: {d}")
# "Priya" is silently lost — only "Rohan" survives

In [ ]:
# ── Mistake 4: Using a mutable key ────────────────────────────────────────────
# Dict keys must be hashable (immutable). Lists are not.

try:
    bad_dict = {[1, 2]: "value"}   # list as key — TypeError
except TypeError as e:
    print(f"TypeError: {e}")

# ✅ Use a tuple instead (tuples are immutable and hashable)
good_dict = {(1, 2): "value"}
print(f"Tuple key works: {good_dict}")

# Real use: (row, col) coordinates as keys
grid = {(0, 0): "A", (0, 1): "B", (1, 0): "C", (1, 1): "D"}
print(f"Grid[0,1]: {grid[(0, 1)]}")

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Word Frequency Counter

Given a sentence, count how many times each word appears (case-insensitive, ignore punctuation). Print the top 5 most frequent words.

In [ ]:
text = """
Data science is the science of extracting knowledge from data.
Data scientists use statistics, machine learning, and data visualization
to understand data and make data-driven decisions.
"""

# Your code here:
# 1. Convert to lowercase, split into words
# 2. Strip punctuation from each word (hint: word.strip('.,\'-'))
# 3. Count with a dict
# 4. Sort by frequency and print top 5


### Exercise 2 — Grade Book

Build a grade book as a `dict` of `{student_name: [list_of_scores]}`. Write three functions:
- `add_score(gradebook, name, score)` — add a score for a student
- `get_average(gradebook, name)` — return that student's average (None if not found)
- `class_report(gradebook)` — print a ranked table of all students with average and letter grade

In [ ]:
def add_score(gradebook, name, score):
    # Your code here
    pass

def get_average(gradebook, name):
    # Your code here
    pass

def class_report(gradebook):
    # Your code here — print ranked table: rank, name, avg, grade
    pass


# Test
gb = {}
add_score(gb, "Priya",  88); add_score(gb, "Priya",  92); add_score(gb, "Priya",  79)
add_score(gb, "Rohan",  55); add_score(gb, "Rohan",  61); add_score(gb, "Rohan",  48)
add_score(gb, "Meera",  95); add_score(gb, "Meera",  88); add_score(gb, "Meera",  91)
add_score(gb, "Karan",  72); add_score(gb, "Karan",  68)
add_score(gb, "Ananya", 33); add_score(gb, "Ananya", 41); add_score(gb, "Ananya", 29)

print(f"Priya's average: {get_average(gb, 'Priya')}")
print(f"Unknown student: {get_average(gb, 'Zara')}")
print()
class_report(gb)

### Exercise 3 — Expense Summary by Category

Given a list of expense records, produce a summary dict `{category: {count, total, avg, max}}`. Then print a ranked summary table.

In [ ]:
expenses = [
    {"date": "2024-01-08", "category": "Food",          "amount": 85},
    {"date": "2024-01-08", "category": "Transport",     "amount": 45},
    {"date": "2024-01-09", "category": "Food",          "amount": 120},
    {"date": "2024-01-09", "category": "Entertainment", "amount": 350},
    {"date": "2024-01-10", "category": "Food",          "amount": 95},
    {"date": "2024-01-10", "category": "Transport",     "amount": 55},
    {"date": "2024-01-10", "category": "Shopping",      "amount": 799},
    {"date": "2024-01-11", "category": "Food",          "amount": 70},
    {"date": "2024-01-11", "category": "Health",        "amount": 200},
    {"date": "2024-01-12", "category": "Transport",     "amount": 40},
    {"date": "2024-01-12", "category": "Food",          "amount": 110},
    {"date": "2024-01-13", "category": "Entertainment", "amount": 550},
    {"date": "2024-01-13", "category": "Food",          "amount": 90},
    {"date": "2024-01-14", "category": "Shopping",      "amount": 299},
    {"date": "2024-01-14", "category": "Transport",     "amount": 50},
]

# Your code here:
# 1. Build summary dict
# 2. Print ranked table: category, count, total (₹), avg (₹), max (₹)
# 3. Show what % of total spend each category represents


### Exercise 4 — Config Reader

A nested config dict stores application settings. Write `get_config(config, key_path, default=None)` where `key_path` is a dot-separated string like `"database.host"` or `"app.features.dark_mode"`.

In [ ]:
config = {
    "app": {
        "name"    : "DataScienceApp",
        "version" : "1.0.0",
        "features": {
            "dark_mode"    : True,
            "notifications": False,
            "language"     : "en-IN",
        },
    },
    "database": {
        "host"    : "localhost",
        "port"    : 5432,
        "name"    : "ds_db",
    },
    "limits": {
        "max_upload_mb" : 50,
        "rate_limit"    : 100,
    },
}

def get_config(config, key_path, default=None):
    """
    Safely retrieve a nested config value using a dot-separated key_path.
    Returns default if any part of the path is missing.

    Example:
        get_config(config, 'database.host') → 'localhost'
        get_config(config, 'app.features.dark_mode') → True
        get_config(config, 'missing.key') → None
    """
    # Your code here
    pass


# Tests
queries = [
    ("app.name",                  "DataScienceApp"),
    ("app.version",               "1.0.0"),
    ("app.features.dark_mode",    True),
    ("app.features.language",     "en-IN"),
    ("database.port",             5432),
    ("limits.max_upload_mb",      50),
    ("missing.key",               None),
    ("app.features.offline_mode", None),
]

print(f"{'Key path':<35}  {'Expected':<20}  {'Got':<20}  {'Pass?'}")
print("─" * 85)
for path, expected in queries:
    result = get_config(config, path)
    passed = result == expected
    print(f"{path:<35}  {str(expected):<20}  {str(result):<20}  {'✓' if passed else '✗'}")

### Exercise 5 — (Challenge) Mini In-Memory Database

Build a simple in-memory database class using a dict as storage. Implement:
- `insert(record)` — add a record (auto-generate an integer ID)
- `find_by_id(id)` — return a record by ID
- `find_where(field, value)` — return all records where field == value
- `update(id, updates)` — merge updates into an existing record
- `delete(id)` — remove a record
- `summary()` — print count and field breakdown

In [ ]:
class SimpleDB:
    def __init__(self):
        self._store   = {}   # id → record
        self._next_id = 1

    def insert(self, record):
        """Add record; assign and return its auto-generated ID."""
        # Your code here
        pass

    def find_by_id(self, record_id):
        """Return record with given ID, or None."""
        pass

    def find_where(self, field, value):
        """Return list of records where record[field] == value."""
        pass

    def update(self, record_id, updates):
        """Merge updates into existing record. Return True if found."""
        pass

    def delete(self, record_id):
        """Delete record. Return True if it existed."""
        pass

    def summary(self):
        """Print record count and unique values for each field."""
        pass


# Test
db = SimpleDB()
db.insert({"name": "Priya",  "dept": "CS", "score": 88})
db.insert({"name": "Rohan",  "dept": "ME", "score": 72})
db.insert({"name": "Meera",  "dept": "CS", "score": 95})
db.insert({"name": "Karan",  "dept": "IT", "score": 61})
db.insert({"name": "Ananya", "dept": "CS", "score": 38})

print("find_by_id(2):",      db.find_by_id(2))
print("find_where dept=CS:", db.find_where("dept", "CS"))

db.update(2, {"score": 85, "city": "Delhi"})
print("After update(2):",    db.find_by_id(2))

db.delete(4)
print("After delete(4):")
db.summary()

---
## 📌 Key Takeaways

- **Always use `.get()` for optional fields, `d[key]` only when you're certain the key exists.** A `KeyError` in production means the key was always optional — someone just didn't design for it. Use `.get(key, default)` to be explicit about what "missing" means for your data.

- **`freq[key] = freq.get(key, 0) + 1` is one of the most important one-liners in Python data science.** It counts frequencies without crashing on the first occurrence. You'll write or recognise this pattern in almost every EDA script, log analyser, and category counter you ever see.

- **A list of dicts is a dataset; a dict of lists is a columnar store.** When you load a CSV into Pandas, it's a dict of lists internally. When you iterate over DataFrame rows, each row is a dict. The mental model transfers directly — master dicts here and DataFrames will feel familiar, not foreign.

---

## What's Next?

**Lesson 07 — Exception Handling**  
Every function in your data toolkit can fail on bad input: a `KeyError` on a missing field, a `ValueError` on a non-numeric string, a `TypeError` on a `None`. Next you'll learn to anticipate these failures gracefully using `try`/`except` — so your pipelines don't crash on the first dirty row.